# Previsão de Câncer por Tecido

**Autores:** Giulia S. Ferreira, Lucas Candinho e Matheus N. Cunha

`RESUMO`

## Dados da atividade

**Nome da Entrega**: Somente o necessário

**Mural de Quests 4**: Jardim do Palácio

# Introdução

O **TCGA** (_The Cancer Genome Atlas_) foi um dos maiores e mais importantes projetos de pesquisa biomédica realizados sobre o cancêr. O objetivo desse projeto era mapear as alterações genômicas, moleculares e histológicas presentes em diferentes tipos de tumores humanos.

O TCGA coletou e analisou mais de 11 mil pacientes, abrangendo 33 tipos de câncer. Dentro do TCGA, algumas subdivisões foram feitas:
- COAD: Colon Adenocarcinoma;
- READ: Rectum Adenocarcinoma;
- STAD: Stomach Adenocarcinoma;
- HNSC: Head and Neck Squamous Cell Carcinoma;
- ESCA: Esophageal Carcinoma.

Neste notebook, desenvolveremos um modelo preditivo que avaliará qual é o tipo de câncer (dentre os 5 citados) com base nos parâmetros físicos e histológicos da amostra, como será aprofundado adiante.

# Importando Módulos

Para realizar a manipulação de dados em _dataframes_ usaremos o módulo `pandas`. Para um suporte estatístico, autilisaremos o módulo `numpy`. 

Na tentativa de traçar a correlação entre o `target` e as `features`, utilizaremos o `OneHotEncoder` do `sklearn.preprocessing`.

Esse codificador também será utilizado na criação de `Pipelines`. Nessa etapa, importaremos a função `Pipeline` do `sklearn.pipeline` para fazer os `Pipelines` em si. Para os modelos, importaremos os seguintes métodos: `StandardScaler`, `ColumnTransformer`, `PCA`, `RFE`, `LogisticRegression`, `KNeighborsClassifier`, `SVC`, `DecisionTreeClassifier`, `RandomForestClassifier`, `GradientBoostingClassifier`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Coleta de Dados

Primeiramente, inserirmos os dados dentro do código, usando do módulo `pandas`.

In [5]:
bacteria = pd.read_csv("Dados/bacteria.unambiguous.decontam.tissue.sample.rpm.relabund.txt", sep="\t", index_col=0).T
metadados = pd.read_csv("Dados/metadata.TCMA.sample.txt", sep="\t")

Podemos, agora, unir os dados em um _dataframe_ só.

In [6]:
df = bacteria.merge(metadados, left_index=True, right_on="bcr_sample_barcode")

Podemos visualizar os nossos dados com o método `<df>.head()`

In [7]:
print(df.head())

     2.0     976.0  1090.0  1117.0    1224.0    1239.0  1297.0   32066.0  \
822  1.0  0.677871     0.0     0.0  0.020165  0.287103     0.0  0.001081   
376  1.0  0.009874     0.0     0.0  0.598349  0.353252     0.0  0.001700   
459  1.0  0.712102     0.0     0.0  0.143843  0.110583     0.0  0.013211   
380  1.0  0.515392     0.0     0.0  0.013983  0.424951     0.0  0.030885   
381  1.0  0.577755     0.0     0.0  0.008072  0.331349     0.0  0.080544   

     40117.0  57723.0  ...  percent_monocyte_infiltration  percent_necrosis  \
822      0.0      0.0  ...                            NaN               5.0   
376      0.0      0.0  ...                            NaN               5.0   
459      0.0      0.0  ...                            NaN               4.0   
380      0.0      0.0  ...                            NaN              11.0   
381      0.0      0.0  ...                            0.0               7.5   

     percent_neutrophil_infiltration  percent_normal_cells  \
822   

# Tratamento de Dados

Nessa seção, realizamos todos os passos de tratamento de dados necessários para utilizarmos esse _dataset_ em nosso modelo.

Antes de tudo, porém, criamos uma cópia do _dataframe_ para mantermos o original intacto.

In [8]:
df_tratado = df.copy(deep=True)

## Removendo colunas com variância 0

Primeiramente removeremos colunas com variância 0, ou seja, colunas que não contribuem para a indução de modelos [1].

In [9]:
num_cols = df.select_dtypes(include=['number'])

variancias = num_cols.var()

colunas_constantes = variancias[variancias == 0].index.tolist()

df_tratado = df_tratado.drop(columns=colunas_constantes)

Perceba que agora temos bem menos colunas (1474 colunas), um fator que facilitará analises posteriores.

In [10]:
print(df_tratado.columns)

Index([                              2.0,                             976.0,
                                  1224.0,                            1239.0,
                                 32066.0,                           74201.0,
                                200795.0,                          200918.0,
                                201174.0,                          203691.0,
       ...
         'percent_monocyte_infiltration',                'percent_necrosis',
       'percent_neutrophil_infiltration',            'percent_normal_cells',
                 'percent_stromal_cells',             'percent_tumor_cells',
                  'percent_tumor_nuclei',                         'project',
                      'HistologicalType',      'ffpe_tumor_slide_submitted'],
      dtype='object', length=1474)


## Análise de Valores Faltantes

Para tratar dos valores faltantes e garantir que o número de valores faltantes, toda coluna que tiver um número de linhas com valores faltantes maior que $437.5$ (que representa 70% do valor total de linhas) será removida do _dataframe_. Inicialmente, vamos analisar quantas delas possuem valores faltantes nessa margem.

In [11]:
linhas_com_nan = df_tratado[df_tratado.isna().any(axis=1)]
soma_nans_por_coluna = linhas_com_nan.isna().sum()
coluna_nans = soma_nans_por_coluna[soma_nans_por_coluna > 437.5].index

soma_nans_por_coluna

biospecimen_sequence          625
composition                   625
current_weight                625
days_to_collection            463
days_to_sample_procurement    625
                             ... 
percent_stromal_cells         330
percent_tumor_cells           320
percent_tumor_nuclei          326
HistologicalType              406
ffpe_tumor_slide_submitted    452
Length: 75, dtype: int64

Agora, vamos remover isso do _dataframe_

In [12]:
df_tratado = df_tratado.drop(columns=coluna_nans)
df_tratado

array(['COAD', 'READ', 'STAD', 'HNSC', 'ESCA'], dtype=object)

## Análise de Colunas com `ints`

O _dataframe_ utilizado foi montado na intenção de comparar a microbiota de diferentes pacientes com diferentes tipos de câncer e comparar com os saudáveis. `< justificativa formal do porquê não iremos usar> `. Como as bactérias estão nomeadas por `ints`, para serem reconhecidas em um outro arquivo, podemos eliminar todas as colunas que possuem um nome como um valor de `int`.

In [ ]:
df_tratado = df_tratado.loc[:, df_tratado.columns.map(lambda x: isinstance(x, str))]

## Determinação do `target` e a escolha das `features`

No _dataframe_ escolhido, a coluna `"project"` será o `target`. Essa coluna categórica identifica de qual subprojeto do TCGA aquela amostra se originou. Abaixo, estão os cinco subprojetos do TCGA, que foram definidos previamente.

In [ ]:
df_tratado["project"].unique()

Agora, iremos determinar quais são as 20 colunas com maior valor de correlação o `target`. Para isso, utilizaremos a função `.corr()` do `pandas`. Essa função mede a associação linear entre duas variáveis numéricas

Essa função só calcula correlação entre valores numéricos, de modo que toda coluna que tiver dados do tipo _object_ devam ser codificadas. Para isso, faremos a codificação do tipo `OneHotEncoder`. Esse codificador retorna uma **matriz esparsa** que, apesar de possuir sua eficiência em memória computacional, para os nossos propósitos, dificulta o trabalho com o `pandas`. Para retornar em um `array`, utilizaremos o argumento `sparse_output=False`.

Além disso, para podermos ter um maior controle no tratamento diferenciado da coluna do `TARGET`, isolaremos essa coluna durante essa etapa.

In [ ]:
TARGET = "project"

series_to_encode = df_tratado[TARGET].to_frame()

OHE = OneHotEncoder(sparse_output=False)
encoded_target = OHE.fit_transform(series_to_encode)

encoded_df = pd.DataFrame(encoded_target, 
                         columns=OHE.get_feature_names_out([TARGET]),
                         index=df_tratado.index)

df_corr = pd.concat([df_tratado, encoded_df], axis=1)

df_corr = df_corr.drop(TARGET, axis=1)

df_corr = pd.get_dummies(df_corr, drop_first=True)

target_columns = [col for col in df_corr.columns if col.startswith(TARGET)]

correlacoes_target = df_corr.corr()[target_columns].abs()
correlacoes_max = correlacoes_target.max(axis=1).sort_values(ascending=False)

colunas_originais = df_tratado.columns.tolist()
colunas_encoded = [col for col in df_corr.columns if col not in colunas_originais]

correlacoes_filtradas = correlacoes_max.drop(colunas_encoded, errors='ignore')

print("25 features mais correlacionadas com o TARGET:")
print(correlacoes_filtradas.head(25))

features_ordenadas = correlacoes_filtradas.index.tolist()

Note que esse tipo de análise avalia correlações lineares, desprezando eventuais correlações não-lineares que possam haver entre o `target` e as `features`. Com base nesses dados, podemos determinar nossos parâmetros.

Além disso, perceba que as colunas `percent_tumor_nuclei` e a `tumor_nuclei_percent` dizem respeito ao mesmo tipo de análise. Por conta disso, eliminaremos a que obteve a menor correlação linear. As colunas que apresentam análises sobre o tipo de amostra (identificando se é um tecido normal ou cancerígeno) também serão eliminadas visto que todas as amostras do `target` são cancerígenas.

Fora as colunas analisadas por meio da correlação linear, a coluna que avalia se o tumor foi extraído por ressecação cirúrgica, a `ProcurementMethod_Resection`, também será utilizada pela sua capacidade de trazer uma relevância na análise do `target

Dessa forma, nossas `features` são:
- `intermediate_dimension`;
- `longest_dimension`;
- `shortest_dimension`;
- `weight`;
- `necrosis_percent`;
- `is_ffpe`;
- `percent_normal_cells`;
- `tumor_nuclei_percent`;
- `percent_tumor_cells`;
- `percent_stromal_cells`;
- `ProcurementMethod_Resection`

In [ ]:
TARGET = "project"

FEATURES = [
    "intermediate_dimension", "longest_dimension", 
    "shortest_dimension", "weight", 
    "necrosis_percent", "is_ffpe", 
    "percent_normal_cells", "tumor_nuclei_percent", 
    "percent_tumor_cells", "percent_stromal_cells"
]

X = df_tratado[TARGET]
y = df_tratado[FEATURES]

# Criação de Pipelines

## Preparação dos Dados

Usaremos as `FEATURES` e o `TARGET`, já definidos na etapa de tratamento de dados, para criarmos os `Pipelines`. Além disso, o código idetificará automaticamente quais das suas *features* são numéricas e quais são categóricas. O que é fundamental para o próximo passo:

O coração da nossa Pipeline de préprocessamento, garantindo que cada coluna receba o tipo de tratamento adequado.

Além disso, a normalização dos dados numéricos será necessária pela exigência de alguns algoritmos, como o `PCA` e o `RFE`.

In [ ]:
NUMERICAL_FEATURES = X.select_dtypes(include=np.number).columns.tolist()
CATEGORICAL_FEATURES = X.select_dtypes(include=['object', 'bool']).columns.tolist()

print(f"Features Numéricas: {NUMERICAL_FEATURES}")
print(f"Features Categóricas: {CATEGORICAL_FEATURES}")

numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler()) 
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 2. Definição do Pré-processamento Base (ColumnTransformer) 
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, NUMERICAL_FEATURES),
        ('cat', categorical_transformer, CATEGORICAL_FEATURES)
    ],
    remainder='passthrough' # "Passa direto" das demais colunas.
)

Em que, 

`numerical_transformer`: É pequena Pipeline que aplica o StandardScaler às colunas numéricas. O StandardScaler padroniza os dados, o que é crucial para modelos como Regressão Logística, KNN e SVM.


`categorical_transformer`: É pequena Pipeline que aplica o OneHotEncoder às colunas categóricas. Este _encoder_ transforma categorias (como 'COAD', 'READ') em colunas binárias (0 ou 1), tornando-as utilizáveis por modelos de ML. O argumento sparse_output=False garante que a saída seja um array denso, mais fácil de manipular.


`ColumnTransformer`: Combina os transformadores. Ele aplica o numerical_transformer apenas às colunas listadas em NUMERICAL_FEATURES e o categorical_transformer apenas às colunas em CATEGORICAL_FEATURES. Isso evita que o scaler seja aplicado em dados categóricos e vice-versa.

 O preprocessor é a primeira etapa de todas as 24 pipelines. Ele garante que os dados estejam limpos e padronizados antes de qualquer outra técnica mais avançada (como PCA ou RFE) ser aplicada. 

## Definição dos Modelos Base e Criação das Pipelines

Este bloco define os modelos que serão testados e os parâmetros iniciais para as técnicas de seleção de features:

`base_models`: Dicionário com os 6 modelos que serão testados.

`N_FEATURES_RFE`: e N_COMPONENTS_PCA: Estes são valores iniciais e arbitrários (30 e 20). O objetivo é que seu colega os otimize. Por exemplo, na otimização, ele testará se 10, 20, 30 ou 40 componentes do PCA são melhores.

Além disso, criaremos as Pipelines, onde o loop for cria as 4 variações de pipeline para cada um dos 6 modelos, totalizando 24 pipelines no dicionário pipelines. 

- Explicando o loop: 

  - {name}_Baseline: É a sua linha de base. O pré-processamento é aplicado, e o modelo é treinado com todas as features resultantes.
  - {name}_PCA [5][6]: Após o pré-processamento, os dados passam pelo PCA, que reduz a dimensionalidade para 20 componentes (inicialmente). O modelo é treinado nessas 20 novas features.
  - {name}_RFE [1][4]: Após o pré-processamento, os dados passam pelo RFE. O RFE usa o próprio modelo (estimator=model) para selecionar as 30 features mais importantes.
  - {name}_PCA_RFE: Esta é a variação mais complexa. Primeiro, o PCA reduz a dimensionalidade, e depois o RFE seleciona as 30 features mais importantes dentro do espaço reduzido pelo PCA. 

  
[3]

In [ ]:
# 3. Definição dos Modelos Base 
base_models = {
    "LogReg": LogisticRegression(max_iter=1000, random_state=33),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(kernel='linear', probability=True, random_state=33),
    "DecisionTree": DecisionTreeClassifier(random_state=33),
    "RandomForest": RandomForestClassifier(random_state=33),
    "GradientBoosting": GradientBoostingClassifier(random_state=33)
}

# 4. Criação das Pipelines Complexas
pipelines = {}
N_FEATURES_RFE = 30 # Número inicial de features para RFE. Deve ser otimizado.
N_COMPONENTS_PCA = 20 # Número inicial de componentes para PCA. Deve ser otimizado.

for name, model in base_models.items():
    # 1. Pipeline Baseline (Apenas Pré-processamento)
    pipelines[f"{name}_Baseline"] = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # 2. Pipeline com PCA (Redução de Dimensionalidade)
    pipelines[f"{name}_PCA"] = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=N_COMPONENTS_PCA, random_state=33)),
        ('classifier', model)
    ])
    
    # 3. Pipeline com RFE (Seleção de Features)
    # Nota: RFE exige um estimador que suporte o atributo 'feature_importances_' ou 'coef_'.
    # Usaremos o modelo base como estimador para RFE.
    # Para modelos que não suportam (como KNN), RFE pode não ser apropriado ou precisaria de um estimador auxiliar.
    # Para simplificar a otimização, vamos usar o modelo base para todos, mas o otimizador deve ser cauteloso com KNN/SVM.
    pipelines[f"{name}_RFE"] = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('rfe', RFE(estimator=model, n_features_to_select=N_FEATURES_RFE)),
        ('classifier', model)
    ])
    
    # 4. Pipeline com PCA + RFE (Combinação)
    pipelines[f"{name}_PCA_RFE"] = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=N_COMPONENTS_PCA, random_state=33)),
        ('rfe', RFE(estimator=model, n_features_to_select=N_FEATURES_RFE)),
        ('classifier', model)
    ])

print(f"Total de {len(pipelines)} pipelines criadas para otimização.")

Agora o dicionário pipelines está pronto para otimização. Isso permitirá que a próxima etapa não apenas otimize os hiperparâmetros internos de cada modelo, mas também encontre os valores ideais para as técnicas de _feature engineering_ (ex: o melhor número de componentes para o `PCA` e o melhor número de features para o `RFE`), garantindo que o modelo final seja o mais robusto e preciso possível.

# Otimização com `Optuna`

# Validação Cruzada

# Treino do Modelo

# Teste e Avaliação do Modelo

# Explicação do Modelo Utilizado

# Conclusão

# XKCD Relevante

# Referências

## _Dataset_ Utilizado

## Gerais

1. Notebook "ATP-203 8.1 - Seleção de atributos" do professor Dr. Daniel R. Cassar.
2. Notebook "ATP-203 7.1 - Dados Sintéticos e Pipeline" do professor Dr. Daniel R. Cassar. 
3. Scikit-learn Developers. Pipeline and Composite Estimators. Scikit-learn Documentation.
Link: https://scikit-learn.org/stable/modules/compose.html
4. Scikit-learn Developers. Recursive feature elimination (RFE). Scikit-learn Documentation.
Link: https://scikit-learn.org/stable/modules/feature_selection.html#recursive-feature-elimination
5. Notebook "ATP-203 8.0" - Redução de Dimensionalidade com PCA" do professor Dr. Daniel R. Cassar.
6. Scikit-learn Developers. Principal component analysis (PCA). Scikit-learn Documentation.
Link: https://scikit-learn.org/stable/modules/decomposition.html#pca
7. 